#  Notebook 02 — Feature Engineering & Business Rationale

##  STAGE 14 — Domain Feature Engineering with Business Rationale
In this notebook, we transform raw customer data into high-predictive domain features. **Every single feature is grounded in an explicit Business Rationale** tied to churn physics, customer friction, or financial return.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = Path("../data/raw")
if not (DATA_PATH / "customer_churn.csv").exists():
    DATA_PATH = Path("data/raw")

df_raw = pd.read_csv(DATA_PATH / "customer_churn.csv")
print(f"Raw Dataset Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

## ️ Feature Construction with Business Rationale

In [ ]:
df_feat = df_raw.copy()

# 1. ContractRisk (Binary Flag)
# Business Rationale: Month-to-Month contracts lack contractual exit barriers, making customers 3x more churn-prone.
df_feat['ContractRisk'] = (df_feat['ContractType'] == 'Month-to-Month').astype(int)

# 2. SupportContactRate (Ticket Density)
# Business Rationale: Support tickets per tenure month measure friction density and unresolved dissatisfaction.
df_feat['SupportContactRate'] = np.round(df_feat['SupportCallsCount'] / (df_feat['Tenure'] + 1), 4)

# 3. Complaint_Severity_Index
# Business Rationale: Combines formal complaint count with low CSAT scores to capture acute dissatisfaction.
df_feat['Complaint_Severity_Index'] = df_feat['ComplaintsCount'] * (6 - df_feat['SatisfactionScore'])

# 4. RevenuePerMonth ($)
# Business Rationale: Measures average monthly account revenue contribution for VIP retention prioritization.
df_feat['RevenuePerMonth'] = np.round(df_feat['TotalCharges'] / (df_feat['Tenure'] + 1e-5), 2)

# 5. Tenure_to_Monthly_Ratio
# Business Rationale: Loyalty relative to monthly cost (price sensitivity index).
df_feat['Tenure_to_Monthly_Ratio'] = np.round(df_feat['Tenure'] / (df_feat['MonthlyCharges'] + 1e-5), 4)

# 6. EngagementScore (Composite Utilization)
# Business Rationale: Aggregates voice, data, and login channels into a unified digital utilization index.
df_feat['EngagementScore'] = np.round(
    (df_feat['CallMinutes'] / 500.0) + (df_feat['DataUsageGB'] / 50.0) + (df_feat['LoginsPerMonth'] / 20.0), 2
)

# 7. CLV_to_Monthly_Ratio
# Business Rationale: Lifetime value multiple relative to monthly cost used in Level 4 ROI maximization.
df_feat['CLV_to_Monthly_Ratio'] = np.round(df_feat['CLV'] / (df_feat['MonthlyCharges'] + 1e-5), 2)

print(f"Features Engineered! Total dataset columns: {df_feat.shape[1]}")
df_feat[['ContractRisk', 'SupportContactRate', 'Complaint_Severity_Index', 'RevenuePerMonth', 'EngagementScore']].head()

##  Feature Correlation with Churn Target

In [ ]:
new_features = ['ContractRisk', 'SupportContactRate', 'Complaint_Severity_Index', 'RevenuePerMonth', 'Tenure_to_Monthly_Ratio', 'EngagementScore', 'CLV_to_Monthly_Ratio', 'Churn']
corr_target = df_feat[new_features].corr()['Churn'].sort_values(ascending=False)
print("Correlation of Engineered Features with Churn:")
print(corr_target)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=corr_target.values[:-1], y=corr_target.index[:-1], palette='coolwarm', ax=ax)
ax.set_title("Engineered Features Correlation with Churn Target", fontsize=13, fontweight='bold')
ax.set_xlabel("Pearson Correlation Coefficient")
plt.tight_layout()
plt.show()

##  Business Rationale Summary Table

| Engineered Feature | Business Rationale | Churn Signal Direction |
| :--- | :--- | :--- |
| **`ContractRisk`** | Month-to-Month contracts lack exit friction. | **Positive Correlation (+)** |
| **`SupportContactRate`** | High support ticket density indicates unresolved service friction. | **Positive Correlation (+)** |
| **`Complaint_Severity_Index`** | Acute dissatisfaction multiplier. | **Positive Correlation (+)** |
| **`Tenure_to_Monthly_Ratio`** | Low ratio highlights price sensitivity in early tenure. | **Negative Correlation (-)** |
| **`EngagementScore`** | Composite multi-channel service engagement. | **Negative Correlation (-)** |

---